# NB11 — 3d Puente al Plan Integrado (TASKS.vsm × comunidades vOS)

**Objetivo:** Indexar tasks pre-voslab (refs del TASKS.vsm como nodos decisión) por conectividad con comunidades vOS del grafo epistémico. Low-connectivity = candidatas a descarte en el triage q1-q8 del plan integrado (`docs/plans/integrated_execution_plan.vsm`).

**NO-NEW-TRUTH:** El grafo no toma el triage — solo lo indexa para S5.

**R10:** Determinista, sin random.

**Denominadores honestos:** total_tasks, matched_tasks, unmatched_tasks, per-status breakdown.

Fase: 3d (plan fase-3 ratified S5 2026-10-11).

In [1]:
import sys
sys.path.insert(0, "../../packages/framework")
from epistemic_graph import build, bridge_view
import json, time

In [2]:
g = build()
t0 = time.time()
v = bridge_view(g)
print(f"bridge: {time.time()-t0:.2f}s")
print(f"graph fingerprint: {v['fingerprint']}")
print(f"schema: {v['schema']}")

bridge: 0.10s
graph fingerprint: ed7ebd5f259e7e20196ae9612a38eb9cde730cbc412edaf41b56185f0141182e
schema: graph_bridge-v1


In [3]:
# denominadores honestos
d = v["denominators"]
print("denominators:")
print(f"  total parsed:    {d['tasks_total_parsed']}")
print(f"  live:            {d['tasks_live']}")
print(f"  matched:         {d['tasks_matched']}")
print(f"  unmatched:       {d['tasks_unmatched']}")
print(f"  live_by_status:  {json.dumps(d['live_by_status'], sort_keys=True)}")
print(f"  communities:     {d['communities_consenso']}")
print(f"  graph_nodes:     {d['graph_nodes']}")
print(f"  graph_edges:     {d['graph_edges']}")

denominators:
  total parsed:    133
  live:            53
  matched:         53
  unmatched:       0
  live_by_status:  {"in_progress": 15, "open": 23, "partial": 2, "queued": 13}
  communities:     66
  graph_nodes:     2150
  graph_edges:     4821


In [4]:
# low-connectivity = candidatas a descarte (triage S5)
low = v["report"]["low_connectivity_candidates"]
print(f"low-connectivity candidates ({len(low)}):")
for ref in low:
    row = next(t for t in v["tasks"] if t["ref"] == ref)
    print(f"  {ref:<45} status={row['status']:<12} conn={row['connectivity']}")

low-connectivity candidates (0):


In [5]:
# connectivity distribution (ascending)
tasks_sorted = sorted(v["tasks"], key=lambda x: x["connectivity"])
print("lowest connectivity (15):")
for t in tasks_sorted[:15]:
    print(f"  {t['ref']:<45} status={t['status']:<12} conn={t['connectivity']}")
print(f"\nconnectivity: min={tasks_sorted[0]['connectivity']} max={tasks_sorted[-1]['connectivity']}")

lowest connectivity (15):
  S1-ISOLATION-RELAY-ARCH                       status=queued       conn=1
  STAGE2C-GENERIC-LIVENESS-SCANNER              status=queued       conn=1
  COMPONENT-B                                   status=in_progress  conn=4
  STAGE2D-HARDWARE-INVENTORY-RECLASSIFICATION   status=queued       conn=5
  C2-METHODOLOGICAL-PARITY-STUDY                status=queued       conn=6
  DASHAI-S4-LAB-EVAL                            status=open         conn=6
  S3-ALLOC-NATS-CRASHLOOP-UNROOTED              status=open         conn=6
  NUCLEUS-REALDATA-WIRING                       status=in_progress  conn=11
  CLIMA-PROCESS-FOOTPRINT-STUDY                 status=open         conn=12
  AC-BATTERY-WATCHDOG-MISSING                   status=open         conn=13
  INFRA-AS-QUERYABLE-DATA-STUDY                 status=queued       conn=14
  BBH-GRANULAR-CLASSIFICATION-INVESTIGATION     status=in_progress  conn=15
  SEC                                           status=partial      c

In [6]:
# high-connectivity = actively integrated into vOS graph
high = v["report"]["high_connectivity"]
print(f"high-connectivity (>=10 shared nodes) ({len(high)}):")
for ref in high[:10]:
    row = next(t for t in v["tasks"] if t["ref"] == ref)
    print(f"  {ref:<45} status={row['status']:<12} conn={row['connectivity']}")
if len(high) > 10:
    print(f"  ... {len(high)-10} more")

high-connectivity (>=10 shared nodes) (46):
  AC-BATTERY-WATCHDOG-MISSING                   status=open         conn=13
  ALGEDONIC-LEDGER-216-TRIAGE-2026-08-03        status=open         conn=129
  AUTOIMMUNE-TEST                               status=open         conn=544
  AXIOM-CANON-CONSISTENCY-2026-08-03            status=open         conn=128
  BBH-ATTACK-SURFACE-EXPANSION-SPEC             status=queued       conn=24
  BBH-BATCH1-REAL-EXECUTION-START               status=in_progress  conn=48
  BBH-C2-TOP-TO-TOP-PIPELINE-DESIGN-NOTE        status=queued       conn=34
  BBH-FLIGHT-PLAN-AND-FULL-SPEC                 status=in_progress  conn=198
  BBH-FLIGHT-PLAN-BUILD                         status=in_progress  conn=72
  BBH-GRANULAR-CLASSIFICATION-INVESTIGATION     status=in_progress  conn=15
  ... 36 more


## Interpretación para S5 (triage q1-q8)

- **low-connectivity (conn=0):** Tasks sin ningún nodo del grafo compartido → **candidatas a descarte**.
- **mid-connectivity (conn 1-10):** Tasks con conexión marginal → revisar en el triage.
- **high-connectivity (conn>=10):** Tasks con fuerte presencia en el grafo vOS → **conservar**.

El grafo NO toma el triage (NO-NEW-TRUTH) — solo indexa. El triage es S5.